# 🪐 Planet Nine search: Stage A, version 2

## How to use this notebook (3 steps)
1. Click **Runtime → Run all**.
2. **Wait about 20–30 minutes.** Keep this tab open and the laptop awake. You will see progress messages.
3. At the **bottom**, read the box called **YOUR RESULTS**. A file called `planet9_results_v2.zip` downloads by itself.

You do **not** need to read or change the code.

## What is new in version 2
* The AI now also learns **how tilted distant orbits naturally are**. In version 1 the practice model assumed
  orbits were flatter than they really are, so it wrongly gave Planet Nine a big tilt.
* Simple ✅ / ⚠️ **exam results** that tell you if the AI can be trusted.
* A **plain-English summary** and **pictures** saved for you.
* If the download from NASA JPL fails, you can upload your old `etno_sample.csv` (folder icon on the left) and run again.

> ⚠️ This is still a **practice model**. Good for learning and testing ideas, not yet a discovery.

In [ ]:
# STEP 0 — install extra packages (about 1–2 minutes)
import importlib, subprocess, sys
for pkg in ["sbi", "rebound", "astropy"]:
    if importlib.util.find_spec(pkg) is None:
        print("Installing", pkg, "...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("✅ Step 0 done: packages ready")

In [ ]:
# SETTINGS — you can change these later for experiments
Q_MIN   = 30.0       # smallest perihelion distance in AU. Experiment later: try 42 (objects far from Neptune)
A_MIN   = 250.0      # smallest semi-major axis in AU
N_SIMS  = 20000      # pretend solar systems used to teach the AI (5000 = faster but less accurate)
EPOCH   = "2026-10-01"

import os, json, time, zipfile, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests, torch
from sbi.inference import NPE
from sbi.utils import BoxUniform
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import (SkyCoord, get_body_barycentric, GeocentricTrueEcliptic,
                                 Galactic, get_constellation)
warnings.filterwarnings("ignore")
torch.manual_seed(42)

C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID, BG = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 11, "axes.titleweight": "bold", "axes.titlesize": 13,
    "legend.frameon": False,
})
SAVED = []          # files that go into the results zip
EXAMS = {}          # exam results for the summary
print("✅ Settings loaded. Q_MIN =", Q_MIN, "AU, N_SIMS =", N_SIMS)

## Step 1 — Download the distant objects ⏱️ a few seconds

In [ ]:
def fetch_etnos(a_min, q_min):
    url = "https://ssd-api.jpl.nasa.gov/sbdb_query.api"
    params = {"fields": "full_name,a,e,i,om,w,q,ad,H,data_arc,n_obs_used",
              "sb-cdata": json.dumps({"AND": [f"a|GT|{a_min}", f"q|GT|{q_min}"]})}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    js = r.json()
    df = pd.DataFrame(js["data"], columns=js["fields"])
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["full_name"] = df["full_name"].str.strip()
    return df

MODE = "REAL"
try:
    allobj = fetch_etnos(A_MIN, 30.0)
    allobj.to_csv("etno_sample.csv", index=False)
    print(f"✅ Downloaded {len(allobj)} objects from NASA JPL")
except Exception as err:
    if os.path.exists("etno_sample.csv"):
        allobj = pd.read_csv("etno_sample.csv")
        MODE = "REAL (saved file)"
        print(f"⚠️ Download failed, so I am using your saved etno_sample.csv ({len(allobj)} objects)")
    else:
        MODE = "MOCK"
        print("⚠️ Download failed and no saved file was found.")
        print("⚠️ MOCK mode: pretend objects will be used. Results are for testing only.")

if MODE != "MOCK":
    keep = (allobj["a"] > A_MIN) & (allobj["q"] > 30.0) & (allobj["data_arc"].fillna(99999) >= 365)
    allobj = allobj[keep].reset_index(drop=True)
    obs = allobj[allobj["q"] > Q_MIN].reset_index(drop=True)
    print(f"✅ Step 1 done: {len(obs)} objects will be used (a > {A_MIN:.0f} AU, q > {Q_MIN:.0f} AU)")

## Step 2 — Prepare the tools ⏱️ instant
Orbit maths and the pretend-solar-system maker. **Nothing to look at here.**

New in version 2: a 7th setting, **natural tilt**, which is how tilted distant orbits are *without* Planet Nine.
The AI learns it from the data, together with the planet.

In [ ]:
D2R = np.pi / 180
def wrap360(x): return np.mod(x, 360.0)

def pole_vector(i_deg, om_deg):
    i, om = np.asarray(i_deg) * D2R, np.asarray(om_deg) * D2R
    return np.stack([np.sin(i) * np.sin(om), -np.sin(i) * np.cos(om), np.cos(i)], axis=-1)

def pole_to_i_om(h):
    h = h / np.linalg.norm(h, axis=-1, keepdims=True)
    return np.arccos(np.clip(h[..., 2], -1, 1)) / D2R, wrap360(np.arctan2(h[..., 0], -h[..., 1]) / D2R)

def circ_mean_R(angles_deg):
    c, s = np.cos(angles_deg * D2R).mean(), np.sin(angles_deg * D2R).mean()
    return wrap360(np.arctan2(s, c) / D2R), np.hypot(c, s)

def circ_diff(a, b):
    return np.abs(((a - b) + 180) % 360 - 180)

def solve_kepler(M, e, n_iter=40):
    E = np.where(e < 0.8, M, np.pi)
    for _ in range(n_iter):
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    return E

def helio_xyz_from_E(a, e, i_deg, om_deg, w_deg, E):
    i, om, w = i_deg * D2R, om_deg * D2R, w_deg * D2R
    xp, yp = a * (np.cos(E) - e), a * np.sqrt(1 - e**2) * np.sin(E)
    cw, sw, co, so, ci, si = np.cos(w), np.sin(w), np.cos(om), np.sin(om), np.cos(i), np.sin(i)
    P = np.stack([cw*co - sw*so*ci, cw*so + sw*co*ci, sw*si])
    Q = np.stack([-sw*co - cw*so*ci, -sw*so + cw*co*ci, cw*si])
    return (xp * P + yp * Q).T

def helio_ecliptic_xyz(a, e, i_deg, om_deg, w_deg, M_deg):
    return helio_xyz_from_E(a, e, i_deg, om_deg, w_deg, solve_kepler(np.mod(M_deg * D2R, 2*np.pi), e))

In [ ]:
PARAM_NAMES = ["mass", "distance", "eccentricity", "planet tilt", "direction", "node", "natural tilt"]
UNITS       = ["Earth masses", "AU", "", "°", "°", "°", "°"]
LOW  = np.array([0.0,  300.0, 0.05,  0.0,   0.0,   0.0,  5.0])
HIGH = np.array([20.0, 1000.0, 0.60, 40.0, 360.0, 360.0, 30.0])
GAL_CROSSINGS = np.array([86.0, 266.0])

def sample_population(n, rng):
    u1, u2, u3 = rng.random(n), rng.random(n), rng.random(n)
    lo, hi = 250.0**-0.5, 1500.0**-0.5
    a = (lo - u1 * (lo - hi)) ** -2
    q = 30.0 + 40.0 * u2
    alpha, H0, H1 = 0.5, 3.0, 8.0
    H = np.log10(10**(alpha*H0) + u3 * (10**(alpha*H1) - 10**(alpha*H0))) / alpha
    return a, np.clip(1 - q / a, 0, 0.995), H

def orbital_angles(theta, a, rng):
    m9, a9, e9, i9, vp9, om9, sig_bg = theta
    n = len(a)
    b9 = a9 * np.sqrt(1 - e9**2)
    F = (m9 / 5.0) * (500.0 / b9) ** 3 * (a / 500.0) ** 1.5
    confined = rng.random(n) < (1 - np.exp(-2 * F))
    inc = np.minimum(np.abs(rng.normal(0, sig_bg, n)), 80)       # NEW: natural tilt is learned
    om = rng.uniform(0, 360, n)
    varpi = rng.uniform(0, 360, n)
    if confined.any():
        k = confined.sum(); Fk = F[confined]
        sig_vp = 20 + 40 / (1 + Fk)
        sig_pole = (6 + 12 / (1 + Fk)) * D2R
        varpi[confined] = wrap360(vp9 + 180 + rng.normal(0, 1, k) * sig_vp)
        h9 = pole_vector(i9, om9)
        ref = np.array([0, 0, 1.0]) if abs(h9[2]) < 0.99 else np.array([1.0, 0, 0])
        e1 = np.cross(h9, ref); e1 /= np.linalg.norm(e1); e2 = np.cross(h9, e1)
        delta = np.abs(rng.normal(0, 1, k)) * sig_pole
        phi = rng.uniform(0, 2*np.pi, k)
        h = (np.cos(delta)[:, None] * h9 + np.sin(delta)[:, None] *
             (np.cos(phi)[:, None] * e1 + np.sin(phi)[:, None] * e2))
        inc[confined], om[confined] = pole_to_i_om(h)
    return inc, om, wrap360(varpi - om), confined

def detection_prob(xyz, H):
    r = np.linalg.norm(xyz, axis=1)
    lam = wrap360(np.arctan2(xyz[:, 1], xyz[:, 0]) / D2R)
    beta = np.arcsin(xyz[:, 2] / r) / D2R
    dgal = np.min(np.abs(((lam[:, None] - GAL_CROSSINGS) + 180) % 360 - 180), axis=1)
    V = H + 10 * np.log10(r)
    Vlim = 24.5 - 2.0 * np.exp(-(dgal / 20) ** 2) - 1.5 * (np.abs(beta) > 25)
    return 1 / (1 + np.exp((V - Vlim) / 0.25))

def sample_time_weighted_E(e, rng):
    E = rng.uniform(0, 2*np.pi, len(e))
    bad = rng.random(len(e)) * (1 + e) > (1 - e * np.cos(E))
    while bad.any():
        E[bad] = rng.uniform(0, 2*np.pi, bad.sum())
        bad[bad] = rng.random(bad.sum()) * (1 + e[bad]) > (1 - e[bad] * np.cos(E[bad]))
    return E

def simulate_detected(theta, n_det, rng, pool=20000, max_rounds=10):
    out, n_got = [], 0
    for _ in range(max_rounds):
        a, e, H = sample_population(pool, rng)
        E = sample_time_weighted_E(e, rng)
        c = H + 10 * np.log10(a * (1 - e * np.cos(E))) < 25.5
        a, e, H, E = a[c], e[c], H[c], E[c]
        inc, om, w, _ = orbital_angles(theta, a, rng)
        det = rng.random(len(a)) < detection_prob(helio_xyz_from_E(a, e, inc, om, w, E), H)
        out.append(np.stack([a[det], e[det], inc[det], om[det], w[det]], axis=1))
        n_got += det.sum()
        if n_got >= n_det:
            break
    return pd.DataFrame(np.concatenate(out)[:n_det], columns=["a", "e", "i", "om", "w"])

def summary_stats(df):
    vp = wrap360(df["om"].values + df["w"].values) * D2R
    om = df["om"].values * D2R
    h = pole_vector(df["i"].values, df["om"].values)
    far = df["a"].values > np.median(df["a"].values)
    hm = h.mean(axis=0)
    return np.array([np.cos(vp).mean(), np.sin(vp).mean(), np.cos(om).mean(), np.sin(om).mean(),
                     hm[0], hm[1], hm[2], 1 - np.linalg.norm(hm),
                     np.cos(vp[far]).mean(), np.sin(vp[far]).mean(), np.median(df["i"].values) / 40.0],
                    dtype=np.float32)

NO_PLANET = np.array([0.0, 500, 0.2, 0, 0, 0, 20.0])
THETA_MOCK = np.array([6.0, 500.0, 0.25, 18.0, 250.0, 100.0, 15.0])
if MODE == "MOCK":
    allobj = simulate_detected(THETA_MOCK, 45, np.random.default_rng(7))
    allobj["q"] = allobj["a"] * (1 - allobj["e"])
    allobj["full_name"] = [f"mock_{k:03d}" for k in range(len(allobj))]
    obs = allobj[allobj["q"] > Q_MIN].reset_index(drop=True)

obs["varpi"] = wrap360(obs["om"] + obs["w"])
N_OBS = len(obs)
x_obs = summary_stats(obs)
print(f"✅ Step 2 done: tools ready, {N_OBS} objects loaded ({MODE})")

## Step 3 — Do the orbits point the same way? Could it be luck? ⏱️ about 1–2 minutes
* **Picture 1:** each arrow is one object. The orange arrow is the average. Longer orange arrow = stronger lean.
* **Picture 2:** thousands of pretend solar systems **without** Planet Nine. How often do they look as bunched as the real ones?
* It also repeats the luck test for stricter choices of objects (far from Neptune).

In [ ]:
def luck_test(df, n_null, rng):
    R = circ_mean_R(wrap360(df["om"].values + df["w"].values))[1]
    Rn = np.array([circ_mean_R(wrap360(d["om"].values + d["w"].values))[1]
                   for d in (simulate_detected(NO_PLANET, len(df), rng) for _ in range(n_null))])
    return R, Rn, (Rn >= R).mean(), n_null

def one_in(p, n_null):
    return f"less than 1 in {n_null}" if p == 0 else f"about 1 in {max(1, round(1 / p))}"

def luck_words(p):
    if p < 0.01: return "very unlikely to be luck"
    if p < 0.05: return "probably not luck"
    if p < 0.20: return "might be luck"
    return "could easily be luck"

vmean, R_obs = circ_mean_R(obs["varpi"].values)
R_obs, R_null, P_LUCK, N_NULL = luck_test(obs, 2000, np.random.default_rng(123))

# Picture 1: compass
fig = plt.figure(figsize=(7, 7.6))
ax = fig.add_subplot(111, projection="polar")
for vp in obs["varpi"].values:
    ax.annotate("", xy=(vp * D2R, 1.0), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=C_BLUE, lw=1.5, alpha=0.75, mutation_scale=11))
ax.annotate("", xy=(vmean * D2R, R_obs), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color=C_ORANGE, lw=4, mutation_scale=20))
ax.text(vmean * D2R, R_obs + 0.14, f"average\n{R_obs:.2f}", color=C_ORANGE, ha="center", fontweight="bold")
ax.set_rticks([]); ax.set_ylim(0, 1.08); ax.grid(color=GRID)
ax.set_thetagrids([0, 90, 180, 270], ["0°", "90°", "180°", "270°"])
fig.suptitle(f"Which way do your {N_OBS} orbits point?", fontsize=15, fontweight="bold", y=0.97)
fig.text(0.5, 0.05, f"Orange arrow: 0 = pointing everywhere, 1 = all the same way. Yours: {R_obs:.2f}", ha="center")
plt.subplots_adjust(top=0.88, bottom=0.12)
plt.savefig("1_orbit_directions.png", dpi=150); SAVED.append("1_orbit_directions.png"); plt.show()

# Picture 2: luck test
fig, ax = plt.subplots(figsize=(9, 4.8))
bins = np.linspace(0, max(0.6, R_obs + 0.1), 49)
_, b, patches = ax.hist(R_null, bins=bins, color=C_AQUA, rwidth=0.9)
for patch, left in zip(patches, b[:-1]):
    if left >= R_obs: patch.set_facecolor(C_BLUE)
ax.axvline(R_obs, color=C_ORANGE, lw=3)
ax.text(R_obs + 0.01, ax.get_ylim()[1] * 0.95, f"your objects ({R_obs:.2f})", color=C_ORANGE, fontweight="bold", va="top")
ax.text(0.98, 0.55, f"{P_LUCK:.0%} of no-planet systems\nlook at least this bunched\n({one_in(P_LUCK, N_NULL)})",
        transform=ax.transAxes, ha="right", color=C_BLUE, fontsize=12, fontweight="bold")
ax.set_yticks([]); ax.spines["left"].set_visible(False)
ax.set_xlabel("how bunched the orbits are (0 = random, 1 = all the same)")
ax.set_title("Could it be luck? 2,000 pretend solar systems with NO Planet Nine", loc="left")
plt.tight_layout(); plt.savefig("2_luck_test.png", dpi=150); SAVED.append("2_luck_test.png"); plt.show()

# Stricter object choices
CUTS = []
for qcut in [30.0, 40.0, 42.0]:
    sub = allobj[allobj["q"] > qcut].reset_index(drop=True)
    if len(sub) < 5: continue
    R_c, _, p_c, n_c = luck_test(sub, 600, np.random.default_rng(int(qcut)))
    CUTS.append((qcut, len(sub), R_c, p_c, n_c))
print(f"✅ Step 3 done: lean = {R_obs:.2f}, luck = {one_in(P_LUCK, N_NULL)} → {luck_words(P_LUCK)}")
for qcut, n, R_c, p_c, n_c in CUTS:
    print(f"   q > {qcut:.0f} AU: {n:3d} objects, lean {R_c:.2f}, luck {one_in(p_c, n_c)}")

## Step 4 — Teach the AI ⏱️ the slow part, about 10–20 minutes
The AI studies thousands of pretend solar systems (with and without Planet Nine) and learns
which kind of planet makes which orbit pattern. Progress is printed below.

In [ ]:
sim_rng = np.random.default_rng(2026)
theta_train = LOW + (HIGH - LOW) * sim_rng.random((N_SIMS, len(LOW)))
x_train = np.zeros((N_SIMS, len(x_obs)), dtype=np.float32)
t0 = time.time()
for k, th in enumerate(theta_train):
    x_train[k] = summary_stats(simulate_detected(th, N_OBS, sim_rng))
    if (k + 1) % max(1, N_SIMS // 10) == 0:
        el = time.time() - t0
        print(f"   {k+1:6d} / {N_SIMS} pretend solar systems made  (about {el/(k+1)*(N_SIMS-k-1)/60:.1f} min left)")
print("   Training the AI now (a few minutes)...")
prior = BoxUniform(low=torch.tensor(LOW, dtype=torch.float32), high=torch.tensor(HIGH, dtype=torch.float32))
inference = NPE(prior=prior, density_estimator="nsf", show_progress_bars=False)   # spline flow: passed the honesty exam in testing
estimator = inference.append_simulations(torch.tensor(theta_train, dtype=torch.float32),
                                         torch.tensor(x_train)).train(training_batch_size=256)
posterior = inference.build_posterior(estimator)
print(f"✅ Step 4 done: AI trained in {(time.time()-t0)/60:.1f} minutes")

def ask_ai(x, n=4000):
    return posterior.sample((n,), x=torch.tensor(x, dtype=torch.float32), show_progress_bars=False).numpy()

## Step 5 — Give the AI two exams ⏱️ about 2–3 minutes
* **Exam 1 (known answers):** we make pretend objects from a planet *we* chose. Does the AI find it?
  One of them has **no planet**. The AI should say the mass could be close to zero.
* **Exam 2 (honesty):** on 200 pretend skies, the true answer should fall inside the AI's
  "likely range" about **68%** of the time. Much less means the AI is overconfident.

In [ ]:
exam_rng = np.random.default_rng(99)
tests = [("planet A (6 Earth masses, 500 AU)", np.array([6.0, 500, 0.25, 18, 250, 100, 15])),
         ("planet B (12 Earth masses, 400 AU)", np.array([12.0, 400, 0.30, 10, 60, 250, 25])),
         ("no planet", np.array([0.0, 600, 0.30, 20, 180, 180, 20]))]
exam1_pass = 0
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (label, th) in zip(axes, tests):
    s = ask_ai(summary_stats(simulate_detected(th, N_OBS, exam_rng)))
    if th[0] == 0:
        ok = np.percentile(s[:, 0], 5) < 1.0
        msg = f"mass range starts at {np.percentile(s[:, 0], 5):.1f}"
    else:
        lo, hi = np.percentile(s[:, 0], [5, 95])
        dir_err = circ_diff(circ_mean_R(s[:, 4])[0], th[4])
        ok = (lo <= th[0] <= hi) and dir_err < 30
        msg = f"mass range {lo:.1f}–{hi:.1f}, direction off by {dir_err:.0f}°"
    exam1_pass += ok
    print(f"   {'✅' if ok else '⚠️'} {label}: {msg}")
    ax.hist(s[:, 0], bins=40, range=(0, 20), color=C_BLUE, rwidth=0.9)
    ax.axvline(th[0], color=C_ORANGE, lw=3)
    ax.set_title(label, fontsize=11); ax.set_xlabel("mass (Earth masses)"); ax.set_yticks([])
fig.suptitle("Exam 1: blue = AI's answer, orange = true answer", fontweight="bold")
plt.tight_layout(); plt.show()
EXAMS["exam1"] = (int(exam1_pass), len(tests))

cov_rng = np.random.default_rng(5)
check_idx = [0, 1, 3, 6]
inside = np.zeros((200, len(check_idx)))
for j in range(200):
    th = LOW + (HIGH - LOW) * cov_rng.random(len(LOW))
    s = ask_ai(summary_stats(simulate_detected(th, N_OBS, cov_rng)), 500)
    for c, k in enumerate(check_idx):
        lo, hi = np.percentile(s[:, k], [16, 84])
        inside[j, c] = lo <= th[k] <= hi
exam2_pass = 0
for c, k in enumerate(check_idx):
    frac = inside[:, c].mean(); ok = 0.55 <= frac <= 0.80
    exam2_pass += ok
    print(f"   {'✅' if ok else '⚠️'} {PARAM_NAMES[k]}: true answer inside likely range {frac:.0%} of the time (goal ≈ 68%)")
EXAMS["exam2"] = (int(exam2_pass), len(check_idx))
print(f"✅ Step 5 done: Exam 1 passed {exam1_pass}/{len(tests)}, Exam 2 passed {exam2_pass}/{len(check_idx)}")

## Step 6 — The AI's answer for the real objects ⏱️ a few seconds
* **Chance of no planet:** before looking at any data, the AI starts at **5%**. If the data push it
  clearly **down**, they prefer a planet. If clearly **up**, they prefer no planet. If it barely moves, the data can't decide.
* Small differences between runs are normal (the AI training has some randomness).
* **Likely range** = where the answer is 68% likely to be (from the 16% value to the 84% value).

In [ ]:
samples = ask_ai(x_obs, 20000)
P_NO = float((samples[:, 0] < 1.0).mean())
PRIOR_NO = 1.0 / 20.0
bf = (P_NO / (1 - P_NO)) / (PRIOR_NO / (1 - PRIOR_NO)) if 0 < P_NO < 1 else (0 if P_NO == 0 else np.inf)
if bf < 1/10:  DECISION = "the data clearly favour a planet"
elif bf < 1/3: DECISION = "the data lean towards a planet"
elif bf <= 3:  DECISION = "the data can't decide yet"
elif bf <= 10: DECISION = "the data lean towards NO planet"
else:          DECISION = "the data clearly favour NO planet"

planet = samples[samples[:, 0] >= 1.0]
def rng_txt(k, fmt="{:.0f}"):
    lo, med, hi = np.percentile(planet[:, k], [16, 50, 84])
    return fmt.format(med), fmt.format(lo), fmt.format(hi)

def edge_note(k):
    hi = np.percentile(planet[:, k], 84); span = HIGH[k] - LOW[k]
    return f"  <- pressing on the {HIGH[k]:.0f} limit, could be even more" if hi > HIGH[k] - 0.1 * span else ""
vp_dir = circ_mean_R(planet[:, 4])[0]

table = pd.DataFrame([(PARAM_NAMES[k], *rng_txt(k, "{:.1f}" if k in (0, 2) else "{:.0f}"), UNITS[k])
                      for k in range(len(LOW)) if k not in (4, 5)],
                     columns=["what", "best guess", "likely from", "likely to", "units"])
print(f"Chance of no planet: before 5% → after {P_NO:.0%}  →  {DECISION}")
print("If a planet exists:"); display(table)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, k in zip(axes.flat, [0, 1, 3, 6]):
    data = samples[:, k] if k in (0, 6) else planet[:, k]
    ax.hist(data, bins=40, range=(LOW[k], HIGH[k]), density=True, color=C_BLUE, rwidth=0.9, label="AI's answer")
    ax.axhline(1 / (HIGH[k] - LOW[k]), color=C_ORANGE, lw=2, ls="--", label="before seeing data")
    if MODE == "MOCK": ax.axvline(THETA_MOCK[k], color=INK, lw=2, label="true (mock)")
    ax.set_title(f"{PARAM_NAMES[k]} ({UNITS[k]})", loc="left"); ax.set_yticks([])
axes[0, 0].legend(loc="upper right")
fig.suptitle(f"What does the AI say?  [{MODE}]", fontweight="bold")
plt.tight_layout(); plt.savefig("4_ai_answer.png", dpi=150); SAVED.append("4_ai_answer.png"); plt.show()
print("✅ Step 6 done")

## Step 7 — Where should we look? ⏱️ about 1 minute
Turns the AI's answer into a sky map, and says how faint and how slow the planet would be.

In [ ]:
EPS = 23.4392911 * D2R
def ecl_to_eq(xyz):
    x, y, z = xyz.T
    return np.stack([x, y*np.cos(EPS) - z*np.sin(EPS), y*np.sin(EPS) + z*np.cos(EPS)], axis=1)

def earth_helio_eq(t):
    return (get_body_barycentric("earth", t) - get_body_barycentric("sun", t)).xyz.to(u.AU).value

def sky_predictions(s, epoch, rng):
    m9, a9, e9, i9, vp9, om9, sig = s.T
    n = len(s); w9 = wrap360(vp9 - om9); M0 = rng.uniform(0, 360, n)
    t0 = Time(epoch); t1 = t0 + 1 * u.day
    pos = {}
    for tag, t, M in [("0", t0, M0), ("1", t1, M0 + 360.0 / (a9**1.5 * 365.25))]:
        helio = ecl_to_eq(helio_ecliptic_xyz(a9, e9, i9, om9, w9, M))
        geo = helio - earth_helio_eq(t); d = np.linalg.norm(geo, axis=1)
        pos["ra"+tag] = wrap360(np.arctan2(geo[:, 1], geo[:, 0]) / D2R); pos["dec"+tag] = np.arcsin(geo[:, 2] / d) / D2R
        if tag == "0": pos["r"], pos["d"] = np.linalg.norm(helio, axis=1), d
    D_km = 2 * 0.808 * m9**0.589 * 6371.0
    albedo = rng.uniform(0.2, 0.75, n)
    V = 5 * np.log10(1329.0 / (D_km * np.sqrt(albedo))) + 5 * np.log10(pos["r"] * pos["d"])
    c0 = SkyCoord(pos["ra0"] * u.deg, pos["dec0"] * u.deg); c1 = SkyCoord(pos["ra1"] * u.deg, pos["dec1"] * u.deg)
    return pd.DataFrame(dict(ra=pos["ra0"], dec=pos["dec0"], r_sun_AU=pos["r"], delta_AU=pos["d"], V_mag=V,
        rate_arcsec_day=c0.separation(c1).arcsec, constellation=get_constellation(c0),
        gal_b=c0.galactic.b.deg, m9=m9, a9=a9, e9=e9, i9=i9, varpi9=vp9, Omega9=om9, natural_tilt=sig,
        albedo=albedo, weight=1.0))

sky = sky_predictions(planet, EPOCH, np.random.default_rng(11))
sky.to_csv("p9_sky_samples.csv", index=False); SAVED.append("p9_sky_samples.csv")
TOP_CONS = sky["constellation"].value_counts(normalize=True).head(3)

def to_moll(ra, dec): return -np.radians(((ra + 180) % 360) - 180), np.radians(dec)
fig = plt.figure(figsize=(11, 9.5))
ax = fig.add_axes([0.04, 0.40, 0.92, 0.52], projection="mollweide")
x, y = to_moll(sky["ra"].values, sky["dec"].values)
ax.hexbin(x, y, gridsize=60, cmap="Blues", mincnt=1, linewidths=0.2, edgecolors=BG)
lon = np.linspace(0, 360, 1441)
for c, col, lab in [(SkyCoord(lon*u.deg, 0*u.deg, frame=GeocentricTrueEcliptic(equinox=Time("J2000"))).icrs, C_AQUA, "path of the Sun and planets"),
                    (SkyCoord(l=lon*u.deg, b=0*u.deg, frame=Galactic).icrs, C_ORANGE, "Milky Way")]:
    xx, yy = to_moll(c.ra.deg, c.dec.deg); j = np.where(np.abs(np.diff(xx)) > np.pi)[0] + 1
    ax.plot(np.insert(xx, j, np.nan), np.insert(yy, j, np.nan), color=col, lw=1.8, label=lab)
ax.set_xticklabels([f"{h}h" for h in [10, 8, 6, 4, 2, 0, 22, 20, 18, 16, 14]], fontsize=9)
ax.grid(color=GRID, lw=0.6)
placed = []
for name in TOP_CONS.index:
    sel = sky[sky["constellation"] == name]
    cra, _ = circ_mean_R(sel["ra"].values); cdec = sel["dec"].median()
    while any(circ_diff(cra, pr) < 25 and abs(cdec - pd_) < 9 for pr, pd_ in placed):
        cdec -= 9                                   # move the label down so names don't overlap
    placed.append((cra, cdec))
    xx, yy = to_moll(np.array([cra]), np.array([cdec]))
    ax.text(xx[0], yy[0], name, fontsize=11, fontweight="bold", ha="center", va="center",
            bbox=dict(facecolor=BG, edgecolor="none", alpha=0.8, pad=1.5))
ax.legend(loc="lower right", bbox_to_anchor=(1.0, -0.12))
ax.set_title(f"If Planet Nine exists, where is it? Darker = more likely  [{MODE}]", pad=16)
ax2 = fig.add_axes([0.10, 0.07, 0.82, 0.22])
ax2.hist(sky["V_mag"], bins=np.arange(17, 28.01, 0.25), color=C_BLUE, rwidth=0.9)
for lim, lab in [(18.0, "good backyard\ncamera (~18)"), (20.5, "ZTF (~20.5)"), (24.5, "Rubin (~24.5)")]:
    ax2.axvline(lim, color=INK2, lw=1.2, ls="--"); ax2.text(lim + 0.12, ax2.get_ylim()[1] * 0.95, lab, fontsize=9, va="top")
ax2.set_yticks([]); ax2.spines["left"].set_visible(False)
ax2.set_xlabel("brightness (magnitude): bigger number = fainter"); ax2.set_title("How faint would it be?", loc="left")
plt.savefig("3_where_to_look.png", dpi=150); SAVED.append("3_where_to_look.png"); plt.show()
print("✅ Step 7 done")

## Step 8 — ⭐ YOUR RESULTS ⭐
Read the box below. The zip file with everything downloads automatically.
If it doesn't, click the **folder icon** on the left and download `planet9_results_v2.zip`.

In [ ]:
if R_obs < 0.3:   lean_words = "weak lean"
elif R_obs < 0.5: lean_words = "medium lean"
else:             lean_words = "strong lean"
V_med = sky["V_mag"].median()
lines = [
    "==================== YOUR RESULTS (version 2) ====================",
    f"Data: {MODE} | {N_OBS} objects | a > {A_MIN:.0f} AU, q > {Q_MIN:.0f} AU",
    "",
    "1) Do the orbits point the same way?",
    f"   Lean strength {R_obs:.2f} (0 = random, 1 = all the same) -> {lean_words}, towards about {vmean:.0f} deg",
    "",
    "2) Could it be luck?",
    f"   {one_in(P_LUCK, N_NULL).capitalize()} no-planet systems look like this -> {luck_words(P_LUCK)}",
    "   (scientists want 1 in 100 or rarer)",
    "",
    "3) Did the AI pass its exams?",
    f"   Exam 1 (known answers): {EXAMS['exam1'][0]}/{EXAMS['exam1'][1]} passed",
    f"   Exam 2 (honesty):       {EXAMS['exam2'][0]}/{EXAMS['exam2'][1]} passed",
    "",
    "4) What does the AI say?",
    f"   Chance of no planet: before 5% -> after {P_NO:.0%} -> {DECISION}",
    "   If a planet exists:",
    "      mass     {} Earth masses (likely {} to {})".format(*rng_txt(0, "{:.1f}")) + edge_note(0),
    "      distance {} AU (likely {} to {})".format(*rng_txt(1)) + edge_note(1),
    "      tilt     {} deg (likely {} to {})".format(*rng_txt(3)),
    f"      direction of its perihelion about {vp_dir:.0f} deg",
    "   Natural tilt of distant orbits: {} deg (likely {} to {})".format(
        *[f"{v:.0f}" for v in np.percentile(samples[:, 6], [50, 16, 84])]),
    "",
    "5) Where to look (if it exists)?",
    "   Most likely constellations: " + ", ".join(f"{c} ({f:.0%})" for c, f in TOP_CONS.items()),
    f"   Brightness about magnitude {V_med:.1f} -> "
    + ("a big survey telescope could see it" if V_med < 21 else "needs a giant telescope like Rubin"),
    f"   Bright enough for Rubin (mag < 24.5) in {(sky['V_mag'] < 24.5).mean():.0%} of cases",
    f"   Moves about {sky['rate_arcsec_day'].median():.1f} arcseconds per day",
    "",
    "6) Stricter choices of objects (luck test only):",
] + [f"   q > {q:.0f} AU: {n} objects, lean {R_c:.2f}, luck {one_in(p_c, n_c)}" for q, n, R_c, p_c, n_c in CUTS] + [
    "",
    "Reminder: practice model. Good for learning, not yet a discovery.",
    "===================================================================",
]
summary_text = "\n".join(lines)
print(summary_text)
with open("results_summary.txt", "w") as f:
    f.write(summary_text + "\n")
files_to_zip = ["results_summary.txt"] + SAVED + (["etno_sample.csv"] if os.path.exists("etno_sample.csv") else [])
with zipfile.ZipFile("planet9_results_v2.zip", "w") as z:
    for fn in files_to_zip:
        z.write(fn)
try:
    from google.colab import files
    files.download("planet9_results_v2.zip")
    print("\n⬇️  Downloading planet9_results_v2.zip ...")
except ImportError:
    print("\n📁 Not in Colab: planet9_results_v2.zip is in the same folder as this notebook.")